In [ ]:
import os
if 'google.colab' in str(get_ipython()):
    # Running in Colab
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = "/content/drive/MyDrive/Teknisk fysik/Utbyte/Kurser/DAML/Project/notebooks"
else:
    # Running locally (Mac/Linux)
    PROJECT_DIR = '/Users/erikdalgard/Library/CloudStorage/GoogleDrive-dalgard.erik@gmail.com/My Drive/Teknisk fysik/Utbyte/Kurser/DAML/Project/notebooks'

os.chdir(PROJECT_DIR)


In [1]:

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import mixed_precision
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger, BackupAndRestore
from sklearn.utils.class_weight import compute_class_weight
from datetime import datetime
import importlib



#Importing modules
import models_2D, models_3D, models_ae

#Importing cutter
import cub_cutter as cc 

#Changing so keras trains with float16 instead of float32 to increase computational speed. 
#mixed_precision.set_global_policy('mixed_float16')

In [5]:
X_train = cc.train_ds

In [ ]:
#Getting 2D CNN models
archi_1_2D = models_2D.get_archi_1_2D()
archi_2_2D = models_2D.get_archi_2_2D()
archi_3_2D = models_2D.get_archi_3_2D()

#Getting 3D CNN models
archi_1_3D = models_3D.get_archi_1_3D()
archi_2_3D = models_3D.get_archi_2_3D()
archi_3_3D = models_3D.get_archi_3_3D()

#Getting autoencoder models
archi_1_ae = models_ae.get_archi_1_AE()
archi_2_ae = models_ae.get_archi_2_AE()
archi_3_ae = models_ae.get_archi_3_AE()

In [ ]:
def train_network(model, X_train, y_train=None, X_val=None, y_val=None, project_dir="", epochs=50, batch_size=64, is_ae=False, is_debug = False):
    """
    Compiles, logs, and trains a given Keras model. Supports both standard classification
    and Autoencoder (AE) reconstruction training.

    Parameters:
        model (keras.Model): The uncompiled Keras model to be trained.
        X_train: Training features.
        y_train: Training labels (Ignored if is_ae=True).
        X_val: Validation features (Optional).
        y_val: Validation labels (Ignored if is_ae=True).
        project_dir (str): Directory for saving assets.
        epochs (int): Maximum number of training iterations.
        batch_size (int): Number of samples per training batch.
        is_ae (bool): Set to True if training an autoencoder.
        is_debug (bool): If set to True, only run 3 epochs on a small set of the data. No data/model is saved
    
    Returns:
        History: The history as a keras object
        checkpoint_path: The file path to the weights of the model
        log_path: The file path to the csv logs of the training


    """

    #If debug is True we only take a small piece of the data to confirm all is working
    if is_debug:
        print("=" * 50)
        print("DEBUG MODE ACTIVE — using subset of data, 3 epochs")
        print("=" * 50)
        X_train = X_train[:128]
        y_train = y_train[:128] if y_train is not None else None
        X_val = X_val[:32] if X_val is not None else None
        y_val = y_val[:32] if y_val is not None else None
        epochs = 3

    # 1. Dynamically configure Loss, Metrics, and Targets based on is_ae flag
    if is_ae:
        print(f"Configuring pipeline for Autoencoder reconstruction task...")
        loss_function = 'mse'
        metrics_list = None  # MSE loss itself acts as the performance tracker for AEs

        # In an autoencoder, the input IS the target output
        train_targets = X_train
        val_targets = X_val if X_val is not None else None
        monitor_metric = 'val_loss'
        monitor_mode = 'min'

        class_weight_dict = None

    else:
        loss_function = 'binary_crossentropy'
        metrics_list = [
            keras.metrics.BinaryAccuracy(name='accuracy'),
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='sensitivity'),
            keras.metrics.AUC(name='auc')
        ]
        train_targets = y_train
        val_targets = y_val
        monitor_metric = 'val_auc'
        monitor_mode = 'max'

        #Calculating class weights to punish positives more than negatives
        classes = np.unique(y_train)
        weights = compute_class_weight('balanced', classes=classes, y=y_train.ravel())
        class_weight_dict = dict(zip(classes, weights))

    # 2. Compile the model with the chosen settings
    model.compile(
        optimizer='adam',
        loss=loss_function,
        metrics=metrics_list
    )

    # 3. Create a clean subfolder dynamically named after the architecture
    model_folder = os.path.join(project_dir, 'training_history')
    history_folder = os.path.join(model_folder, model.name)
    os.makedirs(model_folder, exist_ok=True)
    os.makedirs(history_folder, exist_ok=True)

    time_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    checkpoint_path = os.path.join(history_folder, f'best_model_{time_stamp}.h5')
    log_path = os.path.join(history_folder, f'training_log_{time_stamp}.csv')

    # 4. Setting up automation callbacks
    callbacks = [
        ModelCheckpoint(
            filepath=checkpoint_path,
            monitor=monitor_metric,
            mode=monitor_mode,
            save_best_only=True,
            save_weights_only=False  # Explicitly set this to prevent default leaks
        ),
        EarlyStopping( #If model does not improve after 10 epochs, we stop training and restore best model
            monitor=monitor_metric,
            mode=monitor_mode,
            patience=10,
            restore_best_weights=True
        ),
        CSVLogger(log_path), #logs the training

        ReduceLROnPlateau( #If we stop learning, we wait 5 epochs and then half the learning rate
            monitor=monitor_metric,
            mode=monitor_mode,
            factor=0.5,       
            patience=5,      
            min_lr=1e-6,     
            verbose=1        
        ),
        BackupAndRestore( #If google collabe crashes, this creates a backup folder with the weights of the previous training. If training is succeeded the backup file is deleted.
            backup_dir=os.path.join(history_folder, 'backup')
    ),
    ]

    #No need to save weights and history if we are doing debug.
    if is_debug:
        callbacks = [EarlyStopping(monitor=monitor_metric, mode=monitor_mode, patience=2)]


    # 5. Execute Training
    print(f"Launching training loop for: {model.name}")

    if X_val is not None and val_targets is not None:
        validation_data = (X_val, val_targets)
    else:
        validation_data = None

    history = model.fit(
        X_train, train_targets,
        validation_data=validation_data,
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        class_weight=class_weight_dict,
        verbose=1
    )

    print(f"\nSuccessfully finished training {model.name}!")
    print(f"-> Best weights secured at: {checkpoint_path}")
    print(f"-> History saved to: {log_path}")

    return history, checkpoint_path, log_path

In [ ]:
def train_network1(
    model,
    X_train,
    y_train=None,
    X_val=None,
    y_val=None,
    project_dir="",
    epochs=50,
    batch_size=64,
    is_ae=False,
    is_debug=False
):
    """
    Compiles, logs, and trains a given Keras model.

    Supports:
    - tf.data.Dataset (recommended)
    - NumPy arrays (legacy support)
    - Autoencoder (AE) training
    """

    # =========================
    # DEBUG MODE
    # =========================
    if is_debug:
        print("=" * 50)
        print("DEBUG MODE ACTIVE — using subset of data, 3 epochs")
        print("=" * 50)

        if isinstance(X_train, tf.data.Dataset):
            X_train = X_train.take(32)
            X_val = X_val.take(16) if X_val is not None else None
        else:
            X_train = X_train[:128]
            y_train = y_train[:128] if y_train is not None else None
            X_val = X_val[:32] if X_val is not None else None
            y_val = y_val[:32] if y_val is not None else None

        epochs = 3

    # =========================
    # LOSS + METRICS SETUP
    # =========================
    if is_ae:
        print("Configuring Autoencoder training...")

        loss_function = 'mse'
        metrics_list = None

        train_targets = X_train
        val_targets = X_val

        monitor_metric = 'val_loss'
        monitor_mode = 'min'
        class_weight_dict = None

    else:
        loss_function = 'binary_crossentropy'

        metrics_list = [
            keras.metrics.BinaryAccuracy(name='accuracy'),
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='sensitivity'),
            keras.metrics.AUC(name='auc')
        ]

        train_targets = y_train
        val_targets = y_val

        monitor_metric = 'val_auc'
        monitor_mode = 'max'

        # class weights ONLY if using numpy labels
        class_weight_dict = None
        if y_train is not None and not isinstance(X_train, tf.data.Dataset):
            classes = np.unique(y_train)
            weights = compute_class_weight('balanced', classes=classes, y=y_train.ravel())
            class_weight_dict = dict(zip(classes, weights))

    # =========================
    # COMPILE MODEL
    # =========================
    model.compile(
        optimizer='adam',
        loss=loss_function,
        metrics=metrics_list
    )

    # =========================
    # CREATE LOG DIRECTORIES
    # =========================
    model_folder = os.path.join(project_dir, 'training_history')
    history_folder = os.path.join(model_folder, model.name)

    os.makedirs(model_folder, exist_ok=True)
    os.makedirs(history_folder, exist_ok=True)

    time_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    checkpoint_path = os.path.join(history_folder, f'best_model_{time_stamp}.h5')
    log_path = os.path.join(history_folder, f'training_log_{time_stamp}.csv')

    # =========================
    # CALLBACKS
    # =========================
    callbacks = [
        ModelCheckpoint(
            filepath=checkpoint_path,
            monitor=monitor_metric,
            mode=monitor_mode,
            save_best_only=True,
            save_weights_only=False
        ),
        EarlyStopping(
            monitor=monitor_metric,
            mode=monitor_mode,
            patience=10,
            restore_best_weights=True
        ),
        CSVLogger(log_path),
        ReduceLROnPlateau(
            monitor=monitor_metric,
            mode=monitor_mode,
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=1
        ),
        BackupAndRestore(
            backup_dir=os.path.join(history_folder, 'backup')
        ),
    ]

    if is_debug:
        callbacks = [
            EarlyStopping(
                monitor=monitor_metric,
                mode=monitor_mode,
                patience=2
            )
        ]

    # =========================
    # VALIDATION HANDLING
    # =========================
    if isinstance(X_val, tf.data.Dataset):
        validation_data = X_val
    elif X_val is not None and val_targets is not None:
        validation_data = (X_val, val_targets)
    else:
        validation_data = None

    # =========================
    # TRAINING
    # =========================
    print(f"Launching training loop for: {model.name}")

    if isinstance(X_train, tf.data.Dataset):
        history = model.fit(
            X_train,
            validation_data=validation_data,
            epochs=epochs,
            callbacks=callbacks,
            verbose=1
        )
    else:
        history = model.fit(
            X_train,
            train_targets,
            validation_data=validation_data,
            epochs=epochs,
            batch_size=batch_size,
            callbacks=callbacks,
            class_weight=class_weight_dict,
            verbose=1
        )

    # =========================
    # DONE
    # =========================
    print(f"\nSuccessfully finished training {model.name}!")
    print(f"-> Best weights secured at: {checkpoint_path}")
    print(f"-> History saved to: {log_path}")

    return history, checkpoint_path, log_path

In [ ]:
import numpy as np

print("Generating dummy test dataset...")

# 1. Define sample counts
num_train_samples = 40
num_val_samples = 10

# 2. Shape must match Archi-1: (Height, Width, Slices/Channels)
patch_shape = (20, 20, 6)

# 3. Generate random float values between 0.0 and 1.0 (simulating normalized CT pixels)
X_train_dummy = np.random.rand(num_train_samples, *patch_shape).astype(np.float32)
X_val_dummy = np.random.rand(num_val_samples, *patch_shape).astype(np.float32)

# 4. Generate highly imbalanced binary labels (0 or 1) for classification
# Let's make mostly 0s (healthy) and a couple of 1s (nodules)
y_train_dummy = np.random.choice([0, 1], size=(num_train_samples, 1), p=[0.9, 0.1]).astype(np.float32)
y_val_dummy = np.random.choice([0, 1], size=(num_val_samples, 1), p=[0.9, 0.1]).astype(np.float32)

print(f"-> Dummy Train Features Shape: {X_train_dummy.shape}")
print(f"-> Dummy Train Labels Shape:   {y_train_dummy.shape}")
print(f"-> Dummy Val Features Shape:   {X_val_dummy.shape}")

In [ ]:
train_network(
    model=archi_1_2D,
    X_train=cc.train_ds,
    y_train=y_train_dummy,
    X_val=X_val_dummy,
    y_val=y_val_dummy,
    project_dir=PROJECT_DIR,
    epochs=5,
    batch_size=8,
    is_ae=False,
    is_debug=False
)